<a href="https://colab.research.google.com/github/NayraSousa/mrfi-teste/blob/dev/resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install mrfi
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [3]:
from mrfi import MRFI, EasyConfig
import torch
import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from mrfi.experiment import Acc_experiment, Acc_golden
import math
import random
import csv
from torchvision.models import resnet18
import os


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
def get_mnist(batch_size=64):

  trainset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data',
      train=True,
      download=True,
      transform=transforms.Compose([
          transforms.Resize((32, 32)),
          transforms.ToTensor(),
          transforms.Normalize(mean=(0.1307), std=(0.3081))
      ])
  )
  testset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data',
      train=False,
      download=True,
      transform=transforms.Compose([
          transforms.Resize((32, 32)),
          transforms.ToTensor(),
          transforms.Normalize(mean=(0.1325), std=(0.3105))
      ])
  )

  trainloader = torch.utils.data.DataLoader(
      trainset,
      batch_size=batch_size,
      shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset,
      batch_size=batch_size,
      shuffle=False
  )

  return trainloader, testloader

def get_cifar10(batch_size=64):
  transform = transforms.Compose([
      transforms.ToTensor()])

  trainset = torchvision.datasets.CIFAR10(
      root='/content/drive/MyDrive/LeNet/data', train=True, download=True, transform=transform
  )

  testset = torchvision.datasets.CIFAR10(
      root='/content/drive/MyDrive/LeNet/data', train=False, download=True, transform=transform
  )

  trainloader = torch.utils.data.DataLoader(
      trainset, batch_size=batch_size, shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset, batch_size=batch_size, shuffle=False
  )

  return trainloader, testloader

def load_dataset(dataset, batch_size=64):

  if dataset.lower()=='mnist':
    train, test= get_mnist()

    return train, test

  if dataset.lower()=='cifar10':
    train, test = get_cifar10()

    return train, test

In [5]:
class ResNet18(nn.Module):
  def __init__(self, in_channels, dataset_name, trained=False, num_classes=10):
        super(ResNet18, self).__init__()

        self.backbone = resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Linear(512, num_classes)

        if trained:
          self.load_state_dict(torch.load(f'/content/drive/MyDrive/LeNet/train/resnet18_{dataset_name}.pth'))


  def forward(self, x):
        return self.backbone(x)

  def fit(model, dataset_name, trainloader, epochs=3):
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()

    for epoch in range(epochs):
      running_loss = 0
      batch_size = 100
      for i, data in trainloader:
        inputs, labels = i.to(device), data.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

      print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}")
    torch.save(model.state_dict(), f'/content/drive/MyDrive/LeNet/train/resnet18_{dataset_name}.pth')
    return model

  def test(model, dataset_name, testloader):
    model = model.to(device)
    model.load_state_dict(torch.load(f'/content/drive/MyDrive/LeNet/train/resnet18_{dataset_name}.pth'))
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            break

        print('Accuracy of the network on the 10000 test images: %d %%' % (
          100 * correct / total))

In [9]:
def calculates_number_positions(e, N, t, p=0.5):

  denominador = 1 + (e ** 2) * ((N-1) / ((t**2*p*(1-p))))
  N_inj = N / denominador

  return N_inj

def return_abs_value(model, layer_name, position):

  if layer_name == 'conv1':
    flattened_weights = model.backbone.conv1.weight.data.flatten()
    magnitude = torch.abs(flattened_weights[position]).item()

  if layer_name == 'fc':
    flattened_weights = model.backbone.fc.weight.data.flatten()
    magnitude = torch.abs(flattened_weights[position]).item()

  return magnitude

def return_fi_model(model, layer_name, position, n):

  if layer_name == 'conv1':
    config_str = f"""
                    faultinject:
                      - type: weights
                        name: [weight]
                        quantization:
                          method: SymmericQuantization
                          bit_width: 8
                          dynamic_range: auto
                        selector:
                          method: FixPosition
                          position: {position}
                        error_mode:
                          method: IntFixedBitFlip
                          bit_width: 8
                          bit: {n}
                        module_name: conv1
                    """
  if layer_name == 'fc':
    config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPosition
                            position: {position}
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: {n}
                          module_name: fc
                          """
  econfig = EasyConfig.load_string(config_str)
  fi_model = MRFI(model.eval(), econfig)
  fi_model.to(device)

  return fi_model, econfig

def return_calculation_times(model, layer_name, input_size=28):
  if layer_name == 'conv1':
    OFW = ((input_size + 2*model.backbone.conv1.padding[0] - model.backbone.conv1.kernel_size[0]) // model.backbone.conv1.stride[0]) + 1
    OFH = OFW
    CT_i = OFW * OFH

    return CT_i

  if layer_name == 'fc':
    return 1

def return_gradient_value(model, layer_name):

  if layer_name == 'conv1':
    return model.backbone.conv1.weight.grad.data.flatten()

  if layer_name == 'fc':
    return model.backbone.fc.weight.grad.data.flatten()

In [7]:
def evaluate_with_injection(fi_model, loader):
    fi_model.eval()
    correct_gold = 0
    correct_inj  = 0
    total = 0

    fi_model.observers_reset()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        total += labels.size(0)

        with fi_model.golden_run():
            out_golden = fi_model(images)
            _, pred_g = out_golden.max(dim=1)
            correct_gold += (pred_g == labels).sum().item()

        out_inject = fi_model(images)
        _, pred_f = out_inject.max(dim=1)
        correct_inj += (pred_f == labels).sum().item()

    acc_golden = correct_gold / total
    acc_inject = correct_inj  / total

    return acc_golden, acc_inject

In [10]:
trainloader, testloader = load_dataset('cifar10')
num_params_conv1 = 0
num_params_fc = 0

resnet = ResNet18(3, 'cifar10')
grad_resnet = resnet.fit(dataset_name='cifar10', trainloader=trainloader, epochs=70)
resnet.test('cifar10', testloader)

for name, param in resnet.named_parameters():
  if name == 'backbone.conv1.weight':
      num_params_conv1 = param.count_nonzero() #9408

  if name == 'backbone.fc.weight':
    num_params_fc = param.count_nonzero() #5120

num_params_conv1, num_params_fc

Epoch 1/70, Loss: 1.3327
Epoch 2/70, Loss: 0.8872
Epoch 3/70, Loss: 0.6747
Epoch 4/70, Loss: 0.5249
Epoch 5/70, Loss: 0.3981
Epoch 6/70, Loss: 0.2839
Epoch 7/70, Loss: 0.1968
Epoch 8/70, Loss: 0.1464
Epoch 9/70, Loss: 0.1121
Epoch 10/70, Loss: 0.0957
Epoch 11/70, Loss: 0.0796
Epoch 12/70, Loss: 0.0703
Epoch 13/70, Loss: 0.0690
Epoch 14/70, Loss: 0.0558
Epoch 15/70, Loss: 0.0500
Epoch 16/70, Loss: 0.0559
Epoch 17/70, Loss: 0.0481
Epoch 18/70, Loss: 0.0396
Epoch 19/70, Loss: 0.0434
Epoch 20/70, Loss: 0.0451
Epoch 21/70, Loss: 0.0310
Epoch 22/70, Loss: 0.0374
Epoch 23/70, Loss: 0.0329
Epoch 24/70, Loss: 0.0310
Epoch 25/70, Loss: 0.0260
Epoch 26/70, Loss: 0.0349
Epoch 27/70, Loss: 0.0286
Epoch 28/70, Loss: 0.0252
Epoch 29/70, Loss: 0.0291
Epoch 30/70, Loss: 0.0231
Epoch 31/70, Loss: 0.0219
Epoch 32/70, Loss: 0.0264
Epoch 33/70, Loss: 0.0188
Epoch 34/70, Loss: 0.0236
Epoch 35/70, Loss: 0.0188
Epoch 36/70, Loss: 0.0247
Epoch 37/70, Loss: 0.0202
Epoch 38/70, Loss: 0.0206
Epoch 39/70, Loss: 0.

(tensor(9408, device='cuda:0'), tensor(5120, device='cuda:0'))

In [11]:
num_params_conv1

tensor(9408, device='cuda:0')

In [12]:
# n_conv1 = calculates_number_positions(0.05, num_params_conv1, 2.60)
n_fc3 = calculates_number_positions(0.05, 5120, 2.60)

# pos_conv1 = random.sample(range(3136), int(n_conv1))
# print(pos_conv1)

pos_fc3 = random.sample(range(5120), int(n_fc3))
print(pos_fc3)

[472, 3463, 2818, 1707, 4422, 4976, 4014, 1325, 4185, 2300, 3140, 3950, 889, 2519, 2101, 2656, 2663, 4744, 726, 1521, 2092, 2378, 4251, 4275, 2350, 915, 1866, 840, 3873, 372, 5106, 2008, 931, 4070, 4862, 1566, 2013, 4800, 180, 1585, 4543, 1600, 1196, 4004, 1231, 2754, 4096, 3689, 2919, 2821, 3826, 4867, 3844, 5006, 3951, 434, 1542, 3726, 1200, 1565, 1658, 3966, 3858, 1662, 2328, 3976, 1976, 1235, 2095, 3145, 626, 2050, 9, 2967, 546, 3149, 331, 2014, 2565, 2634, 4216, 3626, 985, 458, 4427, 3884, 1116, 2414, 2957, 1703, 562, 61, 4537, 4725, 4495, 2004, 3580, 3947, 1962, 1699, 2620, 214, 4970, 4787, 4430, 494, 1638, 187, 4702, 1443, 2724, 2171, 857, 4871, 4721, 3057, 1909, 3255, 777, 271, 682, 4200, 2871, 845, 2606, 1021, 2289, 1485, 4454, 4948, 3606, 4221, 4438, 666, 2787, 822, 4556, 3809, 1333, 4898, 594, 3337, 4098, 4771, 3451, 1984, 3872, 1788, 639, 3486, 3797, 54, 3406, 4092, 475, 306, 167, 2446, 1151, 2786, 4712, 5100, 140, 3713, 4806, 1464, 1303, 3000, 946, 3819, 1158, 4242, 4398, 

In [14]:
bits = [7]
layers = ['fc']
metadata = []

# CT_conv1 = return_calculation_times(resnet, layers[0])
CT_fc3 = return_calculation_times(resnet, layers[0])

# gradient_conv1 = return_gradient_value(grad_resnet, layers[0])
gradient_fc3 = return_gradient_value(grad_resnet, layers[0])

file_path = '/content/drive/MyDrive/LeNet/resnet_cifar10_fc_bit7.csv'
write_header = not os.path.exists(file_path)

In [16]:
for n in bits:
  print(n)

  for name in layers:

    print(name)

    # if name == 'conv1':
    #   for position in pos_conv1:

    #     metadata = []

    #     resnet = ResNet18(in_channels=3, dataset_name='cifar10', trained=True)
    #     magnitude = return_abs_value(resnet, name, position)

    #     fi_model, econfig = return_fi_model(resnet, name, position, n)
    #     acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

    #     metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
    #                      'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
    #                      'vulnerability': acc_g-acc_f, 'calculation_times': CT_conv1,
    #                      'gradient': gradient_conv1[position].item()})

    #     with open(file_path, 'a', newline='', encoding='utf-8') as file:
    #       escritor = csv.DictWriter(file, fieldnames=metadata[0].keys())

    #       if write_header:
    #           escritor.writeheader()
    #           write_header = False
    #       escritor.writerows(metadata)

    if name == 'fc':
      for position in pos_fc3:

        metadata = []

        resnet = ResNet18(in_channels=3, dataset_name='cifar10', trained=True)
        magnitude = return_abs_value(resnet, name, position)

        fi_model, econfig = return_fi_model(resnet, name, position, n)
        acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

        metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
                         'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
                         'vulnerability': acc_g-acc_f, 'calculation_times': CT_fc3,
                         'gradient': gradient_fc3[position].item()})

        with open(file_path, 'a', newline='', encoding='utf-8') as file:
          escritor = csv.DictWriter(file, fieldnames=metadata[0].keys())

          if write_header:
              escritor.writeheader()
              write_header = False
          escritor.writerows(metadata)

7
fc


In [ ]:
lenet = ResNet18ForMNIST(trained=True)
config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPositions
                            positions: [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 133, 123, 118, 112, 20, 127, 40, 128, 117, 7, 116]
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: 7
                          module_name: conv1

                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPositions
                            positions: [812, 826, 818, 788, 784, 796, 816, 444, 596, 74, 776, 653, 829, 493, 595, 438, 498, 420, 660, 503, 483, 448, 442, 467, 227]
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: 7
                          module_name: fc3
                          """
econfig = EasyConfig.load_string(config_str)
fi_model = MRFI(lenet.eval(), econfig)
fi_model.to(device)

acc_g, acc_f = evaluate_with_injection(fi_model, testloader)
acc_g, acc_f

(0.9863, 0.2673)

# CONV1

0.9795 [11]

0.9838 [11, 103]

0.9797 [11, 103, 5]

0.9852 [11, 103, 5, 108]

0.9754 [11, 103, 5, 108, 1]

0.9476 [11, 103, 5, 108, 1, 104]

0.973 [11, 103, 5, 108, 1, 104, 107]

0.9084 [11, 103, 5, 108, 1, 104, 107, 10]

0.8319 [11, 103, 5, 108, 1, 104, 107, 10, 6]

0.892 [11, 103, 5, 108, 1, 104, 107, 10, 6, 21]

## Tirando os gradientes negativos em ordem decrescente:

0.8272 [11, 103, 5, 1, 104, 107, 10, 6, 21]

0.7689 [11, 103, 5, 1, 104, 10, 6, 21]

0.8499 [11, 103, 5, 1, 10, 6, 21]

0.9125 [11, 5, 1, 10, 6, 21]

## Botando em ordem crescente de gradient:

0.9125 [5, 10, 1, 6, 21, 11]

## Botando em ordem decrescente de vulnerabilidade:

0.9125 [11, 5, 1, 10, 6, 21]

## Menor acurácia:

Foi dado pelas seguintes posições: [11, 103, 5, 1, 104, 10, 6, 21]. Agora vou testar com essas posições e o restante das 15 mais vulneravéis.

## Adicionando as posições a partir da 11° mais vulnerável:

0.7023 [11, 103, 5, 1, 104, 10, 6, 21, 124]

0.7773 [11, 103, 5, 1, 104, 10, 6, 21, 124, 109]

0.7444  [11, 103, 5, 1, 104, 10, 6, 21, 124, 109, 0]

0.6529 [11, 103, 5, 1, 104, 10, 6, 21, 124, 109, 0, 119]

0.6246 [11, 103, 5, 1, 104, 10, 6, 21, 124, 109, 0, 119, 15]

## Tirando os gradientes negativos em ordem decrescente:

0.5582 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15]

0.6446 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 15]

0.7108  [11, 103, 5, 1, 104, 10, 6, 21, 0, 15]

## Botando em ordem crescente de gradient:

0.8598 [15, 5, 10, 0, 1, 6, 21, 11]

## Botando em ordem crescente de vulnerabilidade:

0.8598 [11, 5, 1, 10, 6, 21, 0, 15]

## Testando a partir do 16° bit mais vulneravel:

0.5333 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 114, 133]

0.5441  [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 114]

0.501 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111]

0.5411 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106]

0.5582 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15]

## Tirando os gradientes negativos em ordem decrescente:

0.4951 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 133]

0.501  [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111]

0.5411 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106]

0.5582 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15]